## 0. 上传并解压项目文件夹（TitanRAG.zip）

本节演示如何将本地 TitanRAG 文件夹压缩为 zip 文件后上传到 Colab，并解压到指定目录。

In [ ]:
# 选择本地 TitanRAG.zip 文件上传
from google.colab import files
uploaded = files.upload()  # 选择 TitanRAG.zip 上传

# 解压到 /content/TitanRAG 目录
import zipfile
import os
os.makedirs('/content/TitanRAG', exist_ok=True)
with zipfile.ZipFile('TitanRAG.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/TitanRAG')

print('解压完成，可在 /content/TitanRAG 下访问所有项目文件')

# 在Colab运行RAG标注脚本的完整流程

本Notebook演示如何将Python脚本转换为Notebook，并在Google Colab中运行，包括依赖安装、路径设置、主流程和结果保存。

## 1. 导入必要的库

首先导入os、sys等Python标准库，以及后续需要的第三方库。

In [ ]:
# 安装依赖（如已安装可跳过）
!pip install torch transformers sentencepiece

import os
import sys
import json

## 2. 设置路径并导入自定义模块

假设你已将 TitanRAG 相关文件夹上传到 Colab 或 Google Drive，请根据实际路径调整 BASE_DIR。

In [1]:
# 按实际路径修改 BASE_DIR
BASE_DIR = '/content/TitanRAG'  # 例如挂载到 Colab 后的路径

sys.path.insert(0, os.path.join(BASE_DIR, 'projects', 'hybrid_titans'))
sys.path.insert(0, os.path.join(BASE_DIR, 'projects', 'hybrid_titans', 'common'))
sys.path.insert(0, os.path.join(BASE_DIR, 'data'))

from embedders import SentenceTransformerEmbedder
from llm_utils import FlanT5Generator
from baselines import PureRAG
import processed_hotpotqa as hotpotqa

NameError: name 'sys' is not defined

## 3. 主处理流程：生成RAG标注数据集

遍历所有问题，调用 PureRAG 检索与生成，保存结果。

In [ ]:
embedder = SentenceTransformerEmbedder(target_dim=256)
llm_generator = FlanT5Generator("google/flan-t5-large")
rag = PureRAG(embedder, llm_generator)

results = []
test_questions = hotpotqa.TEST_QUESTIONS
all_contexts = hotpotqa.get_all_contexts()

all_articles = list(all_contexts.values())
for title, qa_pairs in test_questions.items():
    for question, gt_answer in qa_pairs:
        pred_answer, answer_sentence = rag.answer(question, all_articles)
        if answer_sentence and gt_answer.strip().lower() in answer_sentence.lower():
            label = "可检索"
        else:
            label = "不可检索"
        results.append({
            "question": question,
            "ground_truth": gt_answer,
            "predicted_answer": pred_answer,
            "answer_sentence": answer_sentence,
            "label": label,
            "context": None,
            "title": title
        })

out_path = os.path.join(BASE_DIR, "data/rag_label_hotpot.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"已保存标注数据集到 {out_path}")

## 4. 在Colab中上传和运行Notebook

你可以直接在Colab中上传此 .ipynb 文件并逐步运行每个单元格。

如需上传本地文件到Colab，可使用：

In [ ]:
from google.colab import files
uploaded = files.upload()  # 选择本地 .ipynb 文件上传

# 上传后在Colab中依次运行每个单元格即可